# Fidelity quantum kernel

Build a small feature-map kernel matrix from pairwise state fidelities.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [ ]:
data = np.asarray([[0.1, 0.2, -0.1], [0.7, -0.4, 0.3], [-0.5, 0.6, 0.8], [0.2, 0.9, -0.7]])

def feature_map(values):
    circuit = QuantumCircuit(3)
    for wire, value in enumerate(values):
        circuit.h(wire)
        circuit.rz(float(value), wire)
    for wire in range(2):
        circuit.rzz(float(values[wire] * values[wire + 1]), wire, wire + 1)
    return circuit

circuits = [feature_map(row) for row in data]

def kernel(states):
    return np.asarray([[abs(np.vdot(left, right)) ** 2 for right in states] for left in states])

def reference_kernel():
    return kernel([np.asarray(Statevector.from_instruction(c).data) for c in circuits])

reference, reference_ms, _ = benchmark(reference_kernel)
backend = MettleQBackend(method="statevector", device="cpu")
compiled = [transpile(c, backend, optimization_level=1) for c in circuits]

def mettleq_kernel():
    states = [np.asarray(backend.run(c, shots=1, return_statevector=True).result().data(0)["statevector"]) for c in compiled]
    return kernel(states)

candidate, mettleq_ms, _ = benchmark(mettleq_kernel)
error = max_abs_error(reference, candidate)
method, device = qiskit_selection(backend)
tutorial_result = emit_result(
    notebook="qiskit/13_quantum_kernel.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="fidelity kernel matrix atol=4e-6",
    passed=error <= 4e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_kernel_error": error, "reference": reference, "mettleq": candidate},
)